# Pythia six-run confirmatory reproduction

This is the review-artifact entrypoint for reproducing the paper's six-run held-out-corpus-B experiment.

Default mode is `full_confirmatory`: six Pythia-70M pretraining runs, checkpoint 143000, layer 2, 512 frozen C4-derived token windows, 256 hosts, 5NN/global controls, downstream Fisher/JVP analyses, dose validation, and the two-candidate 1NN-vs-5NN audit inputs.

The artifact packages only frozen token IDs and a small numeric developmental archive; it does **not** package full raw C4 documents or model weights as model inputs. Launch Jupyter from the repository root (or set `SAFE_TRANSFER_ARTIFACT_ROOT`) and run top-to-bottom on a CUDA GPU with Internet access for public model downloads.

In [ ]:
# =========================
# REVIEW-ARTIFACT CONFIG
# =========================
import os, sys, subprocess, importlib.util
from pathlib import Path

MODAL_RUN_MODE = "full_confirmatory"

ARTIFACT_ROOT = Path(os.environ.get("SAFE_TRANSFER_ARTIFACT_ROOT", ".")).resolve()
if not (ARTIFACT_ROOT / "README.md").exists() and (ARTIFACT_ROOT.parent / "README.md").exists():
    ARTIFACT_ROOT = ARTIFACT_ROOT.parent
assert (ARTIFACT_ROOT / "README.md").exists(), (
    "Launch Jupyter from the artifact repository root or set SAFE_TRANSFER_ARTIFACT_ROOT."
)

MODAL_RESULTS_DIR = str(Path(os.environ.get(
    "SAFE_TRANSFER_OUT", ARTIFACT_ROOT / "_model_reruns" / "pythia_confirmatory"
)))
EXISTING_CAUSAL_ZIP_PATH = str(ARTIFACT_ROOT / "inputs" / "developmental_causal_numeric_outputs.zip")
MAIN_RESULTS_ZIP_PATH = str(ARTIFACT_ROOT / "inputs" / "pythia_confirmatory_c4_token_ids.zip")

required = {
    "transformers": "transformers>=4.51,<5",
    "accelerate": "accelerate>=0.30",
    "safetensors": "safetensors",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
}
missing = [spec for mod, spec in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

for _p in [EXISTING_CAUSAL_ZIP_PATH, MAIN_RESULTS_ZIP_PATH]:
    assert Path(_p).exists(), f"Missing packaged artifact input: {_p}"

os.environ["SAFE_TRANSFER_MODE"] = MODAL_RUN_MODE
os.environ["EXISTING_CAUSAL_ZIP"] = EXISTING_CAUSAL_ZIP_PATH
os.environ["MAIN_RESULTS_ZIP"] = MAIN_RESULTS_ZIP_PATH
os.environ["SAFE_TRANSFER_OUT"] = MODAL_RESULTS_DIR
os.environ["MODAL_GPU_MODE"] = "1"

print("Run mode:", MODAL_RUN_MODE)
print("Artifact root:", ARTIFACT_ROOT)
print("Fresh results dir:", MODAL_RESULTS_DIR)
print("Packaged inputs: PASS")

# What Makes a Transformer MLP Update Safely Interchangeable?
## Source compatibility vs downstream visibility vs protocol artifacts

This is a **new falsification-first follow-up** to *Residual Neighbors Are Not Computational Neighbors*.

The previous six-run result established that local MLP donors can be harmful, source-side JVP mismatch is diagnostic beyond update magnitude, and the frozen same-norm **host-tangent repair failed overall (3/6 positive runs)**. We therefore do **not** rescue that result by selecting the three favorable runs or a favorable damage subgroup.

The new question is constructive:

> **Why are some non-native MLP updates tolerated, and can safe transfer be predicted before transplanting the update?**

We make four explanations compete:

- **H0 — magnitude/protocol:** safe donors are merely smaller perturbations or artifacts of the chosen metric/direction.
- **H1 — source-local compatibility:** safe donors agree with the host MLP's local map, \(J_F(m_i)\Delta m\).
- **H2 — downstream visibility:** safe donors can differ substantially, but their difference lies in directions the remaining transformer weakly reads.
- **H3 — portable lexical structure:** same-current-token or same-next-token structure enriches for safe transfer.
- **H4 — equivalence-like structure:** safe transfer is approximately bidirectional/transitive rather than a one-way patching artifact.

### Main new quantity

Let \(G_\ell\) be everything downstream of the patched MLP output. For
\[
\Delta u=u_j-u_i,
\]
we compute the exact directional derivative of final logits
\[
\Delta z=J_G\Delta u.
\]
Raw logit norm is not softmax-invariant, so the primary downstream-visibility scalar is the local Fisher/KL quadratic
\[
Q(\Delta u)=\frac12\Delta z^\top[\mathrm{diag}(p)-pp^\top]\Delta z
=\frac12\operatorname{Var}_{k\sim p}[\Delta z_k].
\]

We also keep the source decomposition
\[
\Delta u=t+e,\qquad t=J_F(m_i)\Delta m,\qquad e=\Delta u-t.
\]

### Anti-post-hoc boundary

- Existing **corpus A** and already-observed repair outcomes are development/exploratory only.
- The new primary downstream-visibility result is frozen on the previously held-out **corpus B** before GPU execution.
- Pair rows estimate within-model effects; the independently pretrained run is the inferential unit.

In [ ]:
from pathlib import Path
import os, io, gc, json, math, random, hashlib, zipfile, warnings, itertools, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from scipy.stats import spearmanr, rankdata, binomtest
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

SEED=20260817
random.seed(SEED); np.random.seed(SEED)

MODE=os.environ.get('SAFE_TRANSFER_MODE','analysis_only')
assert MODE in {'analysis_only','smoke','full_confirmatory'}
RUN_GPU=MODE!='analysis_only'
RUN_RELATION_TESTS=MODE=='full_confirmatory'
RUN_DEVELOPMENTAL=False
RUN_SCALE_CHECKS=bool(int(os.environ.get('SAFE_TRANSFER_SCALE_CHECKS','0')))

EXISTING_CAUSAL_ZIP=Path(os.environ.get(
    'EXISTING_CAUSAL_ZIP','inputs/developmental_causal_numeric_outputs.zip'))
MAIN_RESULTS_ZIP=Path(os.environ.get(
    'MAIN_RESULTS_ZIP','inputs/pythia_confirmatory_c4_token_ids.zip'))

OUT_DIR=Path(os.environ.get('SAFE_TRANSFER_OUT','_model_reruns/pythia_confirmatory'))
OUT_DIR.mkdir(parents=True,exist_ok=True)
for sub in ['per_target','figures','tables','relations','developmental','scope']:
    (OUT_DIR/sub).mkdir(exist_ok=True)

ALL_REPOS=[
    'EleutherAI/pythia-70m',
    'EleutherAI/pythia-70m-seed1',
    'EleutherAI/pythia-70m-seed2',
    'EleutherAI/pythia-70m-seed3',
    'EleutherAI/pythia-70m-seed4',
    'EleutherAI/pythia-70m-seed5',
]
FROZEN_FINAL_REVISIONS={'EleutherAI/pythia-70m': 'de3e4e2d6cbb3b1a51f90fe153ea51fcd8c7c852', 'EleutherAI/pythia-70m-seed1': 'a94dc37ad8326640e664560d32a0d4a229253fce', 'EleutherAI/pythia-70m-seed2': '258e7aedecacc445bdc68229398e49b9e90d59e0', 'EleutherAI/pythia-70m-seed3': 'd6d278fb9b57ed4ce7faf49ad0aae4ee0805936b', 'EleutherAI/pythia-70m-seed4': '3df7f50e40f6af8ff5828969ecf527eb983b83f1', 'EleutherAI/pythia-70m-seed5': '3aadf268d91f74f998fbf12509e60a8a29c7c045'}
PRIMARY_REPOS=ALL_REPOS[:1] if MODE=='smoke' else ALL_REPOS
FINAL_STEP=143000
PRIMARY_LAYER=2
STATE_PCA_DIM=32
MODAL_GPU_MODE=bool(int(os.environ.get('MODAL_GPU_MODE','0')))

# Full confirmatory sample remains frozen at 256 hosts.
POOL_N=128 if MODE=='smoke' else 512
N_CONFIRM=32 if MODE=='smoke' else 256

# Safe defaults for a modest Modal GPU. A later GPU cell may increase these
# if memory is comfortably above ~20 GB.
JVP_BATCH=1 if MODE!='smoke' else 2
PATCH_BATCH=8
CAPTURE_BATCH=16
SOURCE_JVP_BATCH=128

PRIMARY_CONDITION='knn_5'
SAFE_ALPHA_PRIMARY=0.25
SAFE_ALPHA_SWEEP=[0.10,0.25,0.50]
PRIMARY_FISHER_ROT_DEG=30.0
RANDOM_ROTATION_DRAWS=4
DOSE_LAMBDAS=[0.25,0.50,0.75,1.00]
N_TRIANGLES=16 if MODE=='smoke' else 64
DEV_STEPS=[0,4000,143000]
N_DEV=64 if MODE=='smoke' else 128
SCALE_REPOS=['EleutherAI/pythia-160m','EleutherAI/pythia-410m']
RESUME=True
ALLOW_PROTOCOL_CHANGE=False

print('MODE:',MODE)
print('RUN_GPU:',RUN_GPU)
print('OUT_DIR:',OUT_DIR)

## 1. Frozen held-out-B protocol

### Primary endpoint

For each of six independent Pythia-70M runs, final checkpoint, layer 2, held-out corpus B, 5-NN donors:

\[
\rho_r=\rho_{\rm partial}\left(\log(1+Q_{\rm full}),KL\mid
\log\|\Delta u\|,\log\|\Delta m\|,N,A\right).
\]

Success requires **positive \(\rho_r\) in 6/6 runs**. This directly asks whether downstream visibility adds information beyond perturbation size and source-local geometry.

### Gatekept causal test

If the diagnostic passes, test a same-norm **downstream-Fisher rotation**. Define
\[
g=J_G^\top F_pJ_G\Delta u.
\]
Project \(g\) onto the tangent space of the constant-\(\|\Delta u\|\) sphere and rotate \(\Delta u\) by exactly 30 degrees along local steepest descent. Compare the resulting KL against the original donor and same-angle random rotations.

In [ ]:
FROZEN_PROTOCOL={
  'protocol_version':'2026-08-17-safe-transfer-v1',
  'development_data':{
      'corpus':'A / existing causal archive','status':'exploratory_only',
      'reason':'follow-up motivated after observing old repair outcome'},
  'confirmatory_population':{
      'models':ALL_REPOS,'checkpoint':FINAL_STEP,'layer':PRIMARY_LAYER,
      'corpus':'previously held-out corpus B','pool_n':512,'n_hosts':256,
      'state_definition':'actual final-token tensor entering the MLP',
      'donor_rule':'one deterministic donor from first five PCA32 nearest non-self MLP-input states',
      'replication_unit':'independently pretrained Pythia run'},
  'primary_endpoint':{
      'feature':'log1p(downstream_fisher_Q_full)','outcome':'transplant KL',
      'controls':['log exact update displacement','log exact state displacement',
                  'normalized source nonlinear residual','source tangent alignment'],
      'within_run':'partial Spearman via rank residualization',
      'across_run':'exact two-sided sign test','success':'positive in 6/6 runs'},
  'gatekept_causal':{
      'name':'downstream-Fisher same-norm rotation',
      'formula':'g=J_G^T F_p J_G du; rotate du 30deg along -project_sphere(g)',
      'effect':'median(KL_donor-KL_rotation) per run','success':'positive in 6/6',
      'specificity':'same-norm same-angle random rotations'},
  'secondary':['LORO predictive increment','held-out-B source-tangent repair',
      'predicted vs actual repair gain','finite-dose Fisher test','target-NLL VJP',
      'bidirectionality','symmetry/transitivity','same-token/same-next-token controls',
      'developmental drift','optional Pythia scale sanity'],
  'safe_relation':{
      'status':'descriptive_only','primary_alpha':SAFE_ALPHA_PRIMARY,
      'rule':'KL <= alpha * run median global-donor KL AND no top-1 flip',
      'alpha_sweep':SAFE_ALPHA_SWEEP},
  'claim_rules':{
      'no_pairwise_pseudoreplication':True,'no_outcome_selected_primary_subgroup':True,
      'existing_A_repair_gain_is_exploratory':True,'heldout_B_required_for_new_primary':True}
}
protocol_json=json.dumps(FROZEN_PROTOCOL,sort_keys=True,indent=2)
protocol_hash=hashlib.sha256(protocol_json.encode()).hexdigest()
protocol_path=OUT_DIR/'FROZEN_SAFE_TRANSFER_PROTOCOL.json'
if protocol_path.exists():
    old=protocol_path.read_text(); old_hash=hashlib.sha256(old.encode()).hexdigest()
    if old_hash!=protocol_hash and not ALLOW_PROTOCOL_CHANGE:
        raise RuntimeError('Frozen protocol differs from existing OUT_DIR. Use a new output directory.')
else:
    protocol_path.write_text(protocol_json)
print('Protocol SHA256:',protocol_hash)

## 2. Archive integrity + existing result boundary

In [ ]:
def zip_member_ending(zf,suffix):
    hits=[n for n in zf.namelist() if n.endswith(suffix)]
    if len(hits)!=1: raise KeyError((suffix,hits[:5]))
    return hits[0]

def read_csv_ending(path,suffix):
    with zipfile.ZipFile(path) as zf: return pd.read_csv(zf.open(zip_member_ending(zf,suffix)))
def read_npy_ending(path,suffix):
    with zipfile.ZipFile(path) as zf: return np.load(io.BytesIO(zf.read(zip_member_ending(zf,suffix))),allow_pickle=False)
def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''): h.update(chunk)
    return h.hexdigest()

if not EXISTING_CAUSAL_ZIP.exists(): raise FileNotFoundError(EXISTING_CAUSAL_ZIP)
if not MAIN_RESULTS_ZIP.exists(): raise FileNotFoundError(MAIN_RESULTS_ZIP)
old_pairs=read_csv_ending(EXISTING_CAUSAL_ZIP,'PRIMARY_PAIRS_ALL.csv')
old_repairs=read_csv_ending(EXISTING_CAUSAL_ZIP,'PRIMARY_REPAIRS_ALL.csv')
old_run_effects=read_csv_ending(EXISTING_CAUSAL_ZIP,'PRIMARY_RUN_EFFECTS.csv')
old_inference=read_csv_ending(EXISTING_CAUSAL_ZIP,'PRIMARY_INFERENCE.csv')
assert len(old_run_effects)==6
assert old_pairs[['repo','actual_step','condition','source_index']].duplicated().sum()==0
archive_audit=pd.DataFrame([
 {'artifact':'existing pairs','rows':len(old_pairs),'status':'PASS'},
 {'artifact':'existing repairs','rows':len(old_repairs),'status':'PASS'},
 {'artifact':'old run effects','rows':len(old_run_effects),'status':'PASS'}])
archive_audit.to_csv(OUT_DIR/'ARCHIVE_AUDIT.csv',index=False)
display(archive_audit); display(old_inference)

## 3. Existing corpus-A data: descriptive only

This verifies the old failed source-tangent repair and prevents the common mistake of saying "50% of patches worked." The 3/6 value is a **run-level median repair sign**, not an individual-patch success rate.

In [ ]:
old_final=old_pairs[old_pairs.actual_step.eq(FINAL_STEP)].copy()
rows=[]
for repo,g in old_final.groupby('repo'):
    g5=g[g.condition.eq('knn_5')]; gg=g[g.condition.eq('global')]
    gm=float(gg.KL.median())
    row={'repo':repo,'global_KL_median':gm,'knn5_KL_median':float(g5.KL.median()),
         'knn5_no_flip_rate':float((g5.top1_flip==0).mean())}
    for a in SAFE_ALPHA_SWEEP:
        row[f'safe_rate_alpha_{a:g}']=float(((g5.KL<=a*gm)&(g5.top1_flip==0)).mean())
    rows.append(row)
existing_safe=pd.DataFrame(rows)
existing_safe.to_csv(OUT_DIR/'EXPLORATORY_EXISTING_SAFE_RATES.csv',index=False)
display(old_run_effects[['repo','median_KL_reduction','donor_KL','repair_KL','random_KL']])
display(existing_safe)

## 4. Statistics and vector geometry helpers

In [ ]:
def stable_seed(*parts):
    return int(hashlib.sha256('::'.join(map(str,parts)).encode()).hexdigest()[:16],16)%(2**32-1)
def row_norm(x): return np.linalg.norm(np.asarray(x),axis=1)
def safe_unit(x,eps=1e-12):
    x=np.asarray(x); return x/np.maximum(row_norm(x)[:,None],eps)
def partial_spearman(df,x,y,controls):
    cols=[x,y]+list(controls); q=df[cols].replace([np.inf,-np.inf],np.nan).dropna()
    if len(q)<max(20,len(controls)+5): return np.nan
    rx=rankdata(q[x]); ry=rankdata(q[y])
    if controls:
        C=np.column_stack([rankdata(q[c]) for c in controls])
        rx=rx-LinearRegression().fit(C,rx).predict(C)
        ry=ry-LinearRegression().fit(C,ry).predict(C)
    return float(np.corrcoef(rx,ry)[0,1])
def exact_sign_p(values):
    v=np.asarray(values,float); v=v[np.isfinite(v)&(v!=0)]
    if len(v)==0: return np.nan
    return float(binomtest(int((v>0).sum()),len(v),p=.5,alternative='two-sided').pvalue)
def bootstrap_median(values,n_boot=10000,seed=SEED):
    v=np.asarray(values,float); v=v[np.isfinite(v)]
    rng=np.random.default_rng(seed); b=np.median(rng.choice(v,size=(n_boot,len(v)),replace=True),axis=1)
    return tuple(np.quantile(b,[.025,.975]))
def angle_matched_random_rotation(du,target,seed):
    du=np.asarray(du,float); target=np.asarray(target,float); u=safe_unit(du); t=safe_unit(target)
    th=np.arccos(np.clip(np.sum(u*t,axis=1),-1,1)); rng=np.random.default_rng(seed)
    q=rng.normal(size=du.shape); q=q-np.sum(q*u,axis=1,keepdims=True)*u
    bad=row_norm(q)<1e-10
    for idx in np.where(bad)[0]:
        e=np.zeros(du.shape[1]); e[np.argmin(np.abs(u[idx]))]=1.; q[idx]=e-np.dot(e,u[idx])*u[idx]
    q=safe_unit(q); rot=np.cos(th)[:,None]*u+np.sin(th)[:,None]*q
    return row_norm(du)[:,None]*safe_unit(rot)
def rotate_downstream_descent_same_norm(du,grad,angle_deg):
    du=np.asarray(du,float); grad=np.asarray(grad,float); u=safe_unit(du)
    gt=grad-np.sum(grad*u,axis=1,keepdims=True)*u; ok=row_norm(gt)>=1e-10
    gu=safe_unit(gt); th=np.deg2rad(float(angle_deg))
    out=np.cos(th)*u-np.sin(th)*gu; out[~ok]=u[~ok]
    return row_norm(du)[:,None]*safe_unit(out),ok

## 5. Fisher/KL math and autograd unit tests

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as torchF
F=torchF

def fisher_quadratic_from_logit_jvp(base_logits,dz):
    z=np.asarray(base_logits,np.float64); dz=np.asarray(dz,np.float64)
    z=z-z.max(axis=1,keepdims=True); p=np.exp(z); p=p/p.sum(axis=1,keepdims=True)
    mu=np.sum(p*dz,axis=1,keepdims=True); centered=dz-mu
    var=np.sum(p*centered**2,axis=1); q=.5*np.maximum(var,0.)
    return q,p*centered

# Generic differentiable injection toy test.
torch.manual_seed(SEED)
class TinyNet(nn.Module):
    def __init__(self,d=5,v=11):
        super().__init__(); self.mlp=nn.Sequential(nn.Linear(d,9),nn.GELU(),nn.Linear(9,d)); self.head=nn.Linear(d,v)
    def forward(self,x):
        u=self.mlp(x); return self.head((x+u)[:,-1,:])
toy=TinyNet().eval()
for p in toy.parameters(): p.requires_grad_(False)
B,T,D=3,4,5; x=torch.randn(B,T,D); direction=torch.randn(B,D); zero=torch.zeros_like(direction,requires_grad=True)
def toy_logits(delta):
    def hook(module,args,out):
        pad=torch.cat([torch.zeros_like(out[:,:-1,:]),delta[:,None,:]],dim=1); return out+pad
    h=toy.mlp.register_forward_hook(hook)
    try: return toy(x)
    finally: h.remove()
with torch.enable_grad(): base,dz=torch.autograd.functional.jvp(toy_logits,zero,direction,strict=True)
eps=1e-3; fd=(toy_logits(eps*direction)-toy_logits(-eps*direction))/(2*eps)
rel=float(torch.linalg.vector_norm(dz-fd)/torch.clamp(torch.linalg.vector_norm(fd),min=1e-12))
c=torch.randn_like(base); delta=torch.zeros_like(direction,requires_grad=True)
with torch.enable_grad():
    logits=toy_logits(delta); vjp=torch.autograd.grad((logits*c).sum(),delta)[0]
lhs=float((dz*c).sum()); rhs=float((direction*vjp).sum()); dual=abs(lhs-rhs)/max(abs(lhs),abs(rhs),1e-12)
assert rel<5e-3 and dual<1e-5
autodiff_tests=pd.DataFrame([{'test':'JVP finite difference','value':rel,'status':'PASS'},
                             {'test':'JVP/VJP duality','value':dual,'status':'PASS'}])
autodiff_tests.to_csv(OUT_DIR/'AUTODIFF_UNIT_TESTS.csv',index=False); display(autodiff_tests)

## 6. GPU setup and model helpers

In [ ]:
if RUN_GPU:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision('highest')
    DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if DEVICE.type!='cuda': warnings.warn('GPU modes are intended for CUDA.')
    print('DEVICE:',DEVICE)

    if DEVICE.type=='cuda':
        props=torch.cuda.get_device_properties(0)
        _vram_gb=props.total_memory/2**30
        print('GPU:',torch.cuda.get_device_name(0))
        print(f'VRAM: {_vram_gb:.1f} GB')
        # Conservative on <=16 GB; faster on larger GPUs.
        if _vram_gb >= 35:
            JVP_BATCH=4; PATCH_BATCH=32; CAPTURE_BATCH=64; SOURCE_JVP_BATCH=256
        elif _vram_gb >= 20:
            JVP_BATCH=2; PATCH_BATCH=16; CAPTURE_BATCH=32; SOURCE_JVP_BATCH=256
        else:
            JVP_BATCH=1 if MODE!='smoke' else 2
            PATCH_BATCH=8; CAPTURE_BATCH=16; SOURCE_JVP_BATCH=128
        print('Batches:',{
            'JVP_BATCH':JVP_BATCH,
            'PATCH_BATCH':PATCH_BATCH,
            'CAPTURE_BATCH':CAPTURE_BATCH,
            'SOURCE_JVP_BATCH':SOURCE_JVP_BATCH,
        })
    if DEVICE.type=='cuda':
        props=torch.cuda.get_device_properties(0)
        print('GPU:',torch.cuda.get_device_name(0))
        print(f'VRAM: {props.total_memory/2**30:.1f} GB')
        torch.cuda.empty_cache()
else:
    DEVICE=torch.device('cpu'); print('Model loading gated off in analysis_only.')

In [ ]:
if RUN_GPU:
    def stable_seed(*parts):
        text='::'.join(map(str,parts)).encode()
        return int(hashlib.sha256(text).hexdigest()[:16],16)%(2**32-1)

    def clear_memory():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            gc.collect()

    def first_tensor(obj):
        if torch.is_tensor(obj): return obj
        if isinstance(obj,dict):
            for v in obj.values():
                t=first_tensor(v)
                if t is not None: return t
        if isinstance(obj,(tuple,list)):
            for v in obj:
                t=first_tensor(v)
                if t is not None: return t
        return None

    def replace_first_tensor(obj,new_tensor):
        if torch.is_tensor(obj): return new_tensor
        if isinstance(obj,tuple):
            vals=list(obj)
            for i,v in enumerate(vals):
                if first_tensor(v) is not None:
                    vals[i]=replace_first_tensor(v,new_tensor); return type(obj)(vals)
        if isinstance(obj,list):
            vals=list(obj)
            for i,v in enumerate(vals):
                if first_tensor(v) is not None:
                    vals[i]=replace_first_tensor(v,new_tensor); return vals
        if isinstance(obj,dict):
            vals=dict(obj)
            for k,v in vals.items():
                if first_tensor(v) is not None:
                    vals[k]=replace_first_tensor(v,new_tensor); return vals
        raise TypeError('Could not replace first tensor in module output')

    def model_layers(model):
        if hasattr(model,'gpt_neox') and hasattr(model.gpt_neox,'layers'):
            return model.gpt_neox.layers
        if hasattr(model,'model') and hasattr(model.model,'layers'):
            return model.model.layers
        if hasattr(model,'transformer') and hasattr(model.transformer,'h'):
            return model.transformer.h
        raise TypeError('Unsupported causal-LM layer container')

    def layer_mlp(layer):
        if hasattr(layer,'mlp'): return layer.mlp
        raise TypeError('Layer has no .mlp module')

    def input_device(model):
        return next(model.parameters()).device

    def load_model(repo,step):
        revision=FROZEN_FINAL_REVISIONS.get(repo) if step==FINAL_STEP else f'step{step}'
        kwargs=dict(revision=revision,low_cpu_mem_usage=True)
        # Float32 is deliberate for 70m exact-JVP precision; external scale checks use bf16 if available.
        dtype=torch.float32 if 'pythia-70m' in repo else (torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16)
        kwargs['dtype']=dtype
        try: kwargs['attn_implementation']='eager'
        except Exception: pass
        model=AutoModelForCausalLM.from_pretrained(repo,**kwargs).to(DEVICE).eval()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        model.config.use_cache=False
        for p in model.parameters(): p.requires_grad_(False)
        return model

In [ ]:
if RUN_GPU:
    @torch.inference_mode()
    def capture_mlp_io_and_logits(model,windows,layer_index,batch_size=CAPTURE_BATCH):
        mlp=layer_mlp(model_layers(model)[layer_index])
        states=[];updates=[];logits=[]

        def pre_hook(module,args,kwargs):
            x=first_tensor(args)
            if x is None: x=first_tensor(kwargs)
            states.append(x[:,-1,:].detach().float().cpu())

        def out_hook(module,args,output):
            y=first_tensor(output)
            updates.append(y[:,-1,:].detach().float().cpu())

        hp=mlp.register_forward_pre_hook(pre_hook,with_kwargs=True)
        ho=mlp.register_forward_hook(out_hook)
        dev=input_device(model)
        try:
            for start in range(0,len(windows),batch_size):
                ids=torch.as_tensor(windows[start:start+batch_size],dtype=torch.long,device=dev)
                out=model(input_ids=ids,use_cache=False)
                logits.append(out.logits[:,-1,:].detach().float().cpu())
        finally:
            hp.remove(); ho.remove()
        return torch.cat(states).numpy(),torch.cat(updates).numpy(),torch.cat(logits).numpy()

    @torch.inference_mode()
    def eval_mlp_direct(mlp,states_np,batch_size=256):
        dtype=next(mlp.parameters()).dtype
        outputs=[]
        for start in range(0,len(states_np),batch_size):
            x=torch.as_tensor(states_np[start:start+batch_size],dtype=torch.float32,device=DEVICE).to(dtype)
            y=first_tensor(mlp(x))
            outputs.append(y.detach().float().cpu())
        return torch.cat(outputs).numpy()

    @torch.inference_mode()
    def patch_updates(model,windows,next_ids,baseline_logits,layer_index,patch_updates_np,batch_size=PATCH_BATCH):
        mlp=layer_mlp(model_layers(model)[layer_index])
        all_kl=[];all_nll=[];all_flip=[]
        dev=input_device(model)
        for start in range(0,len(windows),batch_size):
            stop=min(start+batch_size,len(windows))
            ids=torch.as_tensor(windows[start:stop],dtype=torch.long,device=dev)
            patch=torch.as_tensor(patch_updates_np[start:stop],dtype=torch.float32)
            def hook(module,args,output,patch=patch):
                y=first_tensor(output); y2=y.clone()
                y2[:,-1,:]=patch.to(device=y.device,dtype=y.dtype)
                return replace_first_tensor(output,y2)
            handle=mlp.register_forward_hook(hook)
            try:
                patched=model(input_ids=ids,use_cache=False).logits[:,-1,:].float()
            finally:
                handle.remove()
            base=torch.as_tensor(baseline_logits[start:stop],dtype=torch.float32,device=patched.device)
            nxt=torch.as_tensor(next_ids[start:stop],dtype=torch.long,device=patched.device)
            logp=F.log_softmax(base,dim=-1); logq=F.log_softmax(patched,dim=-1); p=logp.exp()
            kl=(p*(logp-logq)).sum(-1)
            nll_base=F.cross_entropy(base,nxt,reduction='none')
            nll_patch=F.cross_entropy(patched,nxt,reduction='none')
            flip=(base.argmax(-1)!=patched.argmax(-1)).float()
            all_kl.extend(kl.detach().cpu().tolist())
            all_nll.extend((nll_patch-nll_base).detach().cpu().tolist())
            all_flip.extend(flip.detach().cpu().tolist())
        return np.asarray(all_kl),np.asarray(all_nll),np.asarray(all_flip)

In [ ]:
if RUN_GPU:
    def pythia_mlp_matrices(mlp):
        W1=mlp.dense_h_to_4h.weight.detach().float()
        W2=mlp.dense_4h_to_h.weight.detach().float()
        b1=mlp.dense_h_to_4h.bias.detach().float() if mlp.dense_h_to_4h.bias is not None else None
        return W1,W2,b1

    def activation_derivative(mlp,x_raw):
        W1,_,b1=pythia_mlp_matrices(mlp)
        with torch.enable_grad():
            x=x_raw.detach().float()
            pre=F.linear(x,W1,b1).detach().requires_grad_(True)
            act=mlp.act(pre)
            grad=torch.autograd.grad(act.sum(),pre,create_graph=False,retain_graph=False)[0]
        return grad.detach()

    def exact_pythia_jvp(mlp,host_np,direction_np,batch_size=SOURCE_JVP_BATCH):
        W1,W2,_=pythia_mlp_matrices(mlp); W1=W1.to(DEVICE); W2=W2.to(DEVICE)
        out=[]
        for start in range(0,len(host_np),batch_size):
            host=torch.as_tensor(host_np[start:start+batch_size],dtype=torch.float32,device=DEVICE)
            direction=torch.as_tensor(direction_np[start:start+batch_size],dtype=torch.float32,device=DEVICE)
            gprime=activation_derivative(mlp,host).to(DEVICE)
            hidden_direction=F.linear(direction,W1,bias=None)
            jvp=F.linear(hidden_direction*gprime,W2,bias=None)
            out.append(jvp.detach().cpu())
        return torch.cat(out).numpy()

    def generic_batched_jvp(mlp,host_np,direction_np,batch_size=min(64,SOURCE_JVP_BATCH)):
        """Architecture-neutral fallback. Exact directional derivative of the MLP module."""
        dtype=next(mlp.parameters()).dtype
        out=[]
        for start in range(0,len(host_np),batch_size):
            x=torch.as_tensor(host_np[start:start+batch_size],dtype=torch.float32,device=DEVICE).to(dtype)
            d=torch.as_tensor(direction_np[start:start+batch_size],dtype=torch.float32,device=DEVICE).to(dtype)
            with torch.enable_grad():
                x=x.detach().requires_grad_(True)
                _,jv=torch.autograd.functional.jvp(lambda z:first_tensor(mlp(z)),x,d,create_graph=False,strict=True)
            out.append(jv.detach().float().cpu())
        return torch.cat(out).numpy()

    def exact_source_jvp(mlp,host_np,direction_np):
        if hasattr(mlp,'dense_h_to_4h') and hasattr(mlp,'dense_4h_to_h') and hasattr(mlp,'act'):
            return exact_pythia_jvp(mlp,host_np,direction_np)
        return generic_batched_jvp(mlp,host_np,direction_np)

## 7. Held-out corpus B and fixed confirmatory hosts

In [ ]:
CORPUS_B=read_npy_ending(MAIN_RESULTS_ZIP,'pythia_corpus_B.npy')
CORPUS_B_NEXT=read_npy_ending(MAIN_RESULTS_ZIP,'pythia_corpus_B_next.npy')
assert len(CORPUS_B)>=POOL_N
pool_B=CORPUS_B[:POOL_N]; pool_B_next=CORPUS_B_NEXT[:POOL_N]
rng=np.random.default_rng(stable_seed('heldout-B-hosts'))
HOST_IDX_CONFIRM=np.sort(rng.choice(POOL_N,size=N_CONFIRM,replace=False))
np.save(OUT_DIR/'HELDOUT_B_HOST_INDICES.npy',HOST_IDX_CONFIRM)
print('Frozen B pool:',pool_B.shape,'hosts:',HOST_IDX_CONFIRM.shape)

## 8. Source-pair metrics, donor rules, and relation candidates

In [ ]:
def random_derangement(n,rng):
    idx=np.arange(n)
    for _ in range(1000):
        p=rng.permutation(n)
        if np.all(p!=idx): return p
    return np.roll(idx,1)
def donor_panel_from_states(states,repo,step,layer):
    dim=min(STATE_PCA_DIM,states.shape[1],len(states)-2)
    Z=PCA(n_components=dim,random_state=SEED).fit_transform(states.astype(np.float64))
    inds=NearestNeighbors(n_neighbors=min(7,len(Z))).fit(Z).kneighbors(Z,return_distance=False)[:,1:]
    rng=np.random.default_rng(stable_seed('safe-transfer-donors',repo,step,layer)); k=min(5,inds.shape[1])
    knn1=inds[:,0].astype(int); choice=rng.integers(0,k,size=len(Z)); knn5=inds[np.arange(len(Z)),choice].astype(int)
    return Z,inds[:,:k],{'knn_1':knn1,'knn_5':knn5,'global':random_derangement(len(Z),rng).astype(int)}
def same_group_donor(values,Z,seed):
    values=np.asarray(values); out=np.full(len(values),-1,dtype=int); rng=np.random.default_rng(seed)
    for val in np.unique(values):
        idx=np.where(values==val)[0]
        if len(idx)<2: continue
        for i in idx:
            c=idx[idx!=i]; dist=np.linalg.norm(Z[c]-Z[i],axis=1); tied=c[np.isclose(dist,dist.min())]; out[i]=int(rng.choice(tied))
    return out
def geometry_triangles(knn,hosts,n_triangles,seed):
    adj=[set(map(int,row)) for row in knn]; cand=set()
    for a in map(int,hosts):
        for b in adj[a]:
            for c in adj[a].intersection(adj[b]):
                if len({a,b,c})==3: cand.add(tuple(sorted((a,b,c))))
    cand=sorted(cand)
    if len(cand)<=n_triangles: return cand
    rng=np.random.default_rng(seed); sel=np.sort(rng.choice(len(cand),size=n_triangles,replace=False)); return [cand[i] for i in sel]

if RUN_GPU:
    def source_pair_metrics(states,updates,src,donors,mlp):
        src=np.asarray(src,int); donors=np.asarray(donors,int); dm=states[donors]-states[src]; du=updates[donors]-updates[src]
        ti=exact_source_jvp(mlp,states[src],dm); tj=exact_source_jvp(mlp,states[donors],dm)
        dmn=row_norm(dm); dun=row_norm(du); tin=row_norm(ti); tjn=row_norm(tj); e=du-ti
        df=pd.DataFrame({'source_index':src,'donor_index':donors,'exact_state_displacement':dmn,
          'exact_update_displacement':dun,'normalized_nonlinear_residual':row_norm(e)/np.maximum(dun,1e-12),
          'tangent_alignment':np.sum(ti*du,axis=1)/np.maximum(tin*dun,1e-12),
          'directional_jacobian_gain':tin/np.maximum(dmn,1e-12),
          'endpoint_jacobian_disagreement':row_norm(ti-tj)/np.maximum(dun,1e-12),
          'endpoint_tangent_cosine':np.sum(ti*tj,axis=1)/np.maximum(tin*tjn,1e-12)})
        return df,dm,du,ti,tj,e

## 9. Exact downstream JVP/VJP through the full model suffix

The hook adds an infinitesimal vector to the **native MLP output** at the final token. Differentiating final logits with respect to that vector gives \(J_Gv\). A logit-space covector gives \(J_G^\top c\).

In [ ]:
if RUN_GPU:
    def _forward_logits_with_delta(model,ids,layer_index,delta):
        mlp=layer_mlp(model_layers(model)[layer_index])
        def hook(module,args,output):
            y=first_tensor(output); pad=torch.cat([torch.zeros_like(y[:,:-1,:]),delta[:,None,:].to(y.dtype)],dim=1)
            return replace_first_tensor(output,y+pad)
        h=mlp.register_forward_hook(hook)
        try: return model(input_ids=ids,use_cache=False).logits[:,-1,:].float()
        finally: h.remove()
    def downstream_logit_jvp(model,windows,layer_index,direction_np,batch_size=JVP_BATCH):
        bases=[]; jvps=[]
        for start in range(0,len(windows),batch_size):
            stop=min(start+batch_size,len(windows)); ids=torch.as_tensor(windows[start:stop],dtype=torch.long,device=DEVICE)
            d=torch.as_tensor(direction_np[start:stop],dtype=torch.float32,device=DEVICE); zero=torch.zeros_like(d,requires_grad=True)
            with torch.enable_grad():
                base,dz=torch.autograd.functional.jvp(lambda delta:_forward_logits_with_delta(model,ids,layer_index,delta),zero,d,strict=True)
            bases.append(base.detach().cpu()); jvps.append(dz.detach().cpu())
        return torch.cat(bases).numpy(),torch.cat(jvps).numpy()
    def downstream_vjp_covector(model,windows,layer_index,cov_np,batch_size=JVP_BATCH):
        grads=[]; d=model.config.hidden_size
        for start in range(0,len(windows),batch_size):
            stop=min(start+batch_size,len(windows)); ids=torch.as_tensor(windows[start:stop],dtype=torch.long,device=DEVICE)
            c=torch.as_tensor(cov_np[start:stop],dtype=torch.float32,device=DEVICE)
            delta=torch.zeros((stop-start,d),dtype=torch.float32,device=DEVICE,requires_grad=True)
            with torch.enable_grad():
                logits=_forward_logits_with_delta(model,ids,layer_index,delta); g=torch.autograd.grad((logits*c).sum(),delta)[0]
            grads.append(g.detach().cpu())
        return torch.cat(grads).numpy()

## 10. Unified held-out-B target runner

In [ ]:
if RUN_GPU:
    def target_stem(repo,step,layer,tag): return f"{repo.replace('/','__')}__step{step}__L{layer}__{tag}"
    def run_safe_transfer_target(repo,step,layer,windows_pool,next_pool,hosts,tag='B_confirm',run_interventions=True,run_relations=True):
        stem=target_stem(repo,step,layer,tag); pair_path=OUT_DIR/'per_target'/f'{stem}__pairs.csv'; dose_path=OUT_DIR/'per_target'/f'{stem}__dose.csv'; rel_path=OUT_DIR/'relations'/f'{stem}__relations.csv'; summary_path=OUT_DIR/'per_target'/f'{stem}__summary.json'
        if RESUME and pair_path.exists() and summary_path.exists():
            _cached_summary=json.loads(summary_path.read_text())
            _cached_pairs=pd.read_csv(pair_path)

            # STRICT RESUME GUARD:
            # A smoke result (e.g. 32 hosts / pool 128) must never be reused
            # as a full target (256 hosts / pool 512).
            _expected_hosts=len(np.asarray(hosts))
            _expected_pool=len(np.asarray(windows_pool))
            _cached_hosts=int(_cached_summary.get('n_hosts',-1))
            _cached_pool=int(_cached_summary.get('pool_n',-1))

            if _cached_hosts==_expected_hosts and _cached_pool==_expected_pool:
                print(
                    f'RESUME VALID: {stem} '
                    f'(hosts={_cached_hosts}, pool={_cached_pool})'
                )
                return (
                    _cached_pairs,
                    pd.read_csv(dose_path) if dose_path.exists() else pd.DataFrame(),
                    pd.read_csv(rel_path) if rel_path.exists() else pd.DataFrame(),
                    _cached_summary
                )

            print(
                f'RESUME REJECTED for {stem}: cached hosts/pool='
                f'{_cached_hosts}/{_cached_pool}, expected '
                f'{_expected_hosts}/{_expected_pool}. Recomputing cleanly.'
            )
            for _p in [pair_path,dose_path,rel_path,summary_path]:
                if _p.exists():
                    _p.unlink()
        model=None
        try:
            model=load_model(repo,step); mlp=layer_mlp(model_layers(model)[layer]); windows_pool=np.asarray(windows_pool); next_pool=np.asarray(next_pool); hosts=np.asarray(hosts,int)
            states,updates,base_logits=capture_mlp_io_and_logits(model,windows_pool,layer,batch_size=CAPTURE_BATCH)
            direct=eval_mlp_direct(mlp,states); direct_rel=float(np.linalg.norm(direct-updates)/max(np.linalg.norm(updates),1e-12))
            if direct_rel>=5e-3: raise RuntimeError(f'Direct MLP reproduction error {direct_rel}')
            Z,knn,sets=donor_panel_from_states(states,repo,step,layer)
            sk,sn,sf=patch_updates(model,windows_pool[hosts],next_pool[hosts],base_logits[hosts],layer,updates[hosts])
            if float(sk.mean())>2e-5 or float(sk.max())>1e-3: raise RuntimeError(f'Self patch failed {sk.mean()} {sk.max()}')
            sets['same_current_token']=same_group_donor(windows_pool[:,-1],Z,stable_seed('same-current',repo,step,layer))
            sets['same_next_token']=same_group_donor(next_pool,Z,stable_seed('same-next',repo,step,layer))
            pair_parts=[]; dose_parts=[]
            for cond in ['knn_1','knn_5','global','same_current_token','same_next_token']:
                dmapping=sets[cond]; valid=dmapping[hosts]>=0
                if not np.any(valid): continue
                s=hosts[valid]; donors=dmapping[s].astype(int); w=windows_pool[s]; nxt=next_pool[s]; base=base_logits[s]; ui=updates[s]; uj=updates[donors]
                KL,NLL,FLIP=patch_updates(model,w,nxt,base,layer,uj)
                met,dm,du,ti,tj,e=source_pair_metrics(states,updates,s,donors,mlp)
                met['condition']=cond; met['KL']=KL; met['NLL_delta']=NLL; met['top1_flip']=FLIP; met['repo']=repo; met['actual_step']=step; met['layer']=layer; met['corpus']=tag; met['state_distance_pca']=row_norm(Z[donors]-Z[s]); met['same_current_token']=windows_pool[s,-1]==windows_pool[donors,-1]; met['same_next_token']=next_pool[s]==next_pool[donors]
                do_downstream=(cond in {PRIMARY_CONDITION,'knn_1'})
                if do_downstream:
                    base0,dz=downstream_logit_jvp(model,w,layer,du); _,dzt=downstream_logit_jvp(model,w,layer,ti)

                    # IMPORTANT: `base` was captured with CAPTURE_BATCH, whereas `base0`
                    # is recomputed inside the differentiable JVP wrapper with JVP_BATCH.
                    # On a T4, different GEMM batch shapes can change raw logits slightly
                    # even when the probability distribution is effectively identical.
                    # Therefore the wrapper sanity check must be done in probability/KL
                    # space, not with a raw max-logit threshold.
                    b0=torch.as_tensor(base0,dtype=torch.float64)
                    bb=torch.as_tensor(base,dtype=torch.float64)
                    logp0=F.log_softmax(b0,dim=-1)
                    logpb=F.log_softmax(bb,dim=-1)
                    p0=logp0.exp()
                    wrapper_KL=(p0*(logp0-logpb)).sum(-1).cpu().numpy()
                    wrapper_max_KL=float(np.max(wrapper_KL))
                    wrapper_mean_KL=float(np.mean(wrapper_KL))
                    raw_max_logit_diff=float(np.max(np.abs(base0-base)))

                    print(
                        f'JVP zero-delta sanity: raw max logit diff={raw_max_logit_diff:.3e}, '
                        f'mean KL={wrapper_mean_KL:.3e}, max KL={wrapper_max_KL:.3e}'
                    )

                    # Same behavioral floor used by the functional self-patch sanity check.
                    # If this fails, stop: the differentiable wrapper is changing model behavior.
                    if wrapper_mean_KL>2e-5 or wrapper_max_KL>1e-3:
                        raise RuntimeError(
                            f'Zero-delta JVP wrapper changed output distribution: '
                            f'mean KL={wrapper_mean_KL:.3e}, max KL={wrapper_max_KL:.3e}, '
                            f'raw max logit diff={raw_max_logit_diff:.3e}'
                        )

                    # Use the baseline returned by the SAME differentiable forward pass
                    # for Fisher/VJP quantities, so the local derivative and p(z) are
                    # evaluated at exactly the same numerical base point.
                    base_jvp=base0
                    dze=dz-dzt
                    q,cov=fisher_quadratic_from_logit_jvp(base_jvp,dz)
                    qt,_=fisher_quadratic_from_logit_jvp(base_jvp,dzt)
                    qe,_=fisher_quadratic_from_logit_jvp(base_jvp,dze)
                    dun=row_norm(du); tin=row_norm(ti); en=row_norm(e)
                    met['downstream_fisher_Q_full']=q; met['downstream_fisher_Q_tangent']=qt; met['downstream_fisher_Q_residual']=qe; met['downstream_fisher_gain_full']=np.sqrt(2*q)/np.maximum(dun,1e-12); met['downstream_fisher_gain_residual']=np.sqrt(2*qe)/np.maximum(en,1e-12); met['downstream_residual_fraction_Q']=qe/np.maximum(q,1e-12); met['downstream_logit_jvp_norm']=row_norm(dz)
                    gf=downstream_vjp_covector(model,w,layer,cov); met['fisher_gn_grad_norm']=row_norm(gf); met['du_fisher_grad_alignment']=np.sum(safe_unit(du)*safe_unit(gf),axis=1)
                    z=base_jvp.astype(np.float64); z=z-z.max(axis=1,keepdims=True); p=np.exp(z); p=p/p.sum(axis=1,keepdims=True); cn=p.copy(); cn[np.arange(len(s)),nxt.astype(int)]-=1.; gn=downstream_vjp_covector(model,w,layer,cn); met['nll_vjp_norm']=row_norm(gn); met['predicted_first_order_NLL_delta']=np.sum(gn*du,axis=1)
                    rep=dun[:,None]*safe_unit(ti); ktr,ntr,ftr=patch_updates(model,w,nxt,base,layer,ui+rep); met['KL_source_tangent_repair']=ktr; met['source_tangent_repair_gain']=KL-ktr
                    scale=dun/np.maximum(tin,1e-12); qtr,_=fisher_quadratic_from_logit_jvp(base_jvp,dzt*scale[:,None]); met['predicted_Q_source_tangent_repair']=qtr; met['predicted_source_tangent_repair_gain']=q-qtr
                    revK,revN,revF=patch_updates(model,windows_pool[donors],next_pool[donors],base_logits[donors],layer,updates[s]); met['reverse_KL']=revK; met['reverse_NLL_delta']=revN; met['reverse_top1_flip']=revF
                    if run_interventions:
                        fdisp,eligible=rotate_downstream_descent_same_norm(du,gf,PRIMARY_FISHER_ROT_DEG); kf,nf,ff=patch_updates(model,w,nxt,base,layer,ui+fdisp); met['fisher_rotation_eligible']=eligible; met['KL_fisher_rotation']=kf; met['NLL_fisher_rotation']=nf; met['flip_fisher_rotation']=ff; met['fisher_rotation_gain']=KL-kf
                        rks=[]
                        for draw in range(RANDOM_ROTATION_DRAWS):
                            rd=angle_matched_random_rotation(du,fdisp,stable_seed('fisher-random',repo,step,layer,draw)); kr,_,_=patch_updates(model,w,nxt,base,layer,ui+rd); rks.append(kr)
                        rks=np.stack(rks,axis=1); met['KL_angle_matched_random_mean']=rks.mean(axis=1); met['fisher_vs_random_gain']=rks.mean(axis=1)-kf
                        for lam in DOSE_LAMBDAS:
                            kd,nd,fd=patch_updates(model,w,nxt,base,layer,ui+float(lam)*du); dose_parts.append(pd.DataFrame({'repo':repo,'actual_step':step,'layer':layer,'corpus':tag,'condition':cond,'source_index':s,'donor_index':donors,'lambda':lam,'KL':kd,'NLL_delta':nd,'top1_flip':fd,'predicted_Fisher_KL':(float(lam)**2)*q}))
                pair_parts.append(met)
            pair_df=pd.concat(pair_parts,ignore_index=True); dose_df=pd.concat(dose_parts,ignore_index=True) if dose_parts else pd.DataFrame(); rel_df=pd.DataFrame()
            if run_relations:
                tris=geometry_triangles(knn,hosts,N_TRIANGLES,stable_seed('triangles',repo,step,layer)); edges=[]
                for tid,(a,b,c) in enumerate(tris):
                    for eid,(ss,dd) in enumerate([(a,b),(b,a),(a,c),(c,a),(b,c),(c,b)]): edges.append((tid,eid,ss,dd))
                if edges:
                    tid=np.array([e0 for e0,_,_,_ in edges]); eid=np.array([e1 for _,e1,_,_ in edges]); ss=np.array([e2 for _,_,e2,_ in edges],int); dd=np.array([e3 for _,_,_,e3 in edges],int)
                    kr,nr,fr=patch_updates(model,windows_pool[ss],next_pool[ss],base_logits[ss],layer,updates[dd]); rel_df=pd.DataFrame({'repo':repo,'actual_step':step,'layer':layer,'corpus':tag,'triangle_id':tid,'edge_id':eid,'source_index':ss,'donor_index':dd,'KL':kr,'NLL_delta':nr,'top1_flip':fr})
            pair_df.to_csv(pair_path,index=False); 
            if len(dose_df): dose_df.to_csv(dose_path,index=False)
            if len(rel_df): rel_df.to_csv(rel_path,index=False)
            summary={'repo':repo,'actual_step':step,'layer':layer,'corpus':tag,'pool_n':len(windows_pool),'n_hosts':len(hosts),'direct_mlp_relative_error':direct_rel,'self_patch_mean_KL':float(sk.mean()),'self_patch_max_KL':float(sk.max()),'model_commit_hash':getattr(model.config,'_commit_hash',None)}; summary_path.write_text(json.dumps(summary,indent=2))
            return pair_df,dose_df,rel_df,summary
        finally:
            if model is not None: del model
            clear_memory()

In [ ]:
# =========================
# FULL CONFIRMATORY PREFLIGHT
# =========================
assert MODE == "full_confirmatory", f"For the review-artifact rerun use full_confirmatory, got {MODE!r}"
assert len(PRIMARY_REPOS) == 6, PRIMARY_REPOS
assert N_CONFIRM == 256, N_CONFIRM
assert POOL_N == 512, POOL_N

print("PRE-FLIGHT PASS")
print("  mode:", MODE)
print("  independent runs:", len(PRIMARY_REPOS))
print("  hosts/run:", N_CONFIRM)
print("  donor pool/run:", POOL_N)
print("  checkpoint:", FINAL_STEP)
print("  layer:", PRIMARY_LAYER)
print("  output dir:", OUT_DIR)
print("\nExpected next header: CONFIRMATORY 1/6")

## 11. Run held-out-B confirmatory suite

### T4 execution note

`full_confirmatory` is intentionally **sequential**: only one Pythia-70M checkpoint is resident on the T4 at a time.  
The notebook writes a `per_target/...csv` and summary after each completed model. If Colab disconnects, rerun from the top with the same Drive folder; `RESUME=True` skips completed targets.

Do not lower the full confirmatory `N_CONFIRM=256` just to make a favorable result cheaper. If you need a quick code check, use `smoke`; its statistics are not the paper result.

In [ ]:
confirm_pairs=confirm_dose=confirm_relations=None
if RUN_GPU:
    pp=[]; dd=[]; rr=[]; ss=[]
    for run_no,repo in enumerate(PRIMARY_REPOS,1):
        _t0=time.time()
        print(f'\n=== CONFIRMATORY {run_no}/{len(PRIMARY_REPOS)}: {repo} ===')
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        p,d,r,s=run_safe_transfer_target(repo,FINAL_STEP,PRIMARY_LAYER,pool_B,pool_B_next,HOST_IDX_CONFIRM,'B_confirm',True,RUN_RELATION_TESTS)
        pp.append(p); ss.append(s)
        if len(d): dd.append(d)
        if len(r): rr.append(r)
        elapsed=(time.time()-_t0)/60
        if torch.cuda.is_available():
            peak=torch.cuda.max_memory_allocated()/2**30
            print(f'Finished {repo} in {elapsed:.1f} min | peak allocated VRAM {peak:.2f} GB')
        else:
            print(f'Finished {repo} in {elapsed:.1f} min')
    confirm_pairs=pd.concat(pp,ignore_index=True); confirm_dose=pd.concat(dd,ignore_index=True) if dd else pd.DataFrame(); confirm_relations=pd.concat(rr,ignore_index=True) if rr else pd.DataFrame()
    confirm_pairs.to_csv(OUT_DIR/'CONFIRMATORY_B_PAIRS.csv',index=False)
    if len(confirm_dose): confirm_dose.to_csv(OUT_DIR/'CONFIRMATORY_B_DOSE.csv',index=False)
    if len(confirm_relations): confirm_relations.to_csv(OUT_DIR/'CONFIRMATORY_B_RELATION_EDGES.csv',index=False)
    pd.DataFrame(ss).to_csv(OUT_DIR/'CONFIRMATORY_B_TARGET_SUMMARIES.csv',index=False)
else: print('Held-out-B GPU suite pending.')

## 12. Primary downstream-visibility inference

In [ ]:
CONTROLS=['log_update_distance','log_state_distance','normalized_nonlinear_residual','tangent_alignment']
def prep(q):
    q=q.copy(); q['log_update_distance']=np.log1p(q.exact_update_displacement.clip(lower=0)); q['log_state_distance']=np.log1p(q.exact_state_displacement.clip(lower=0)); q['log_downstream_Q']=np.log1p(q.downstream_fisher_Q_full.clip(lower=0)); return q
if confirm_pairs is None and (OUT_DIR/'CONFIRMATORY_B_PAIRS.csv').exists(): confirm_pairs=pd.read_csv(OUT_DIR/'CONFIRMATORY_B_PAIRS.csv')
if confirm_pairs is None:
    primary_new=pd.DataFrame([{'status':'PENDING_HELDOUT_B_GPU_RUN'}]); display(primary_new)
else:
    q=prep(confirm_pairs[confirm_pairs.condition.eq(PRIMARY_CONDITION)]); rows=[]
    for repo,g in q.groupby('repo'):
        rows.append({'repo':repo,'n':len(g),'partial_rho_Q_to_KL':partial_spearman(g,'log_downstream_Q','KL',CONTROLS),'partial_rho_gain_to_KL':partial_spearman(g,'downstream_fisher_gain_full','KL',CONTROLS)})
    runp=pd.DataFrame(rows); vals=runp.partial_rho_Q_to_KL.to_numpy(); ci=bootstrap_median(vals); npos=int((vals>0).sum())
    primary_new=pd.DataFrame([{'status':'COMPLETE','n_runs':len(vals),'positive_runs':npos,'median_partial_rho':float(np.nanmedian(vals)),'bootstrap95_low':ci[0],'bootstrap95_high':ci[1],'sign_p':exact_sign_p(vals),'primary_success_6of6':bool(len(vals)==6 and npos==6)}]); runp.to_csv(OUT_DIR/'PRIMARY_DOWNSTREAM_VISIBILITY_RUNS.csv',index=False); primary_new.to_csv(OUT_DIR/'PRIMARY_DOWNSTREAM_VISIBILITY_INFERENCE.csv',index=False); display(runp); display(primary_new)

## 13. Leave-one-run-out prediction: magnitude vs source vs downstream

In [ ]:
FEATURES={
    'magnitude_only':['log_update_distance'],
    'source_geometry':[
        'log_update_distance','log_state_distance',
        'normalized_nonlinear_residual','tangent_alignment',
        'endpoint_jacobian_disagreement'
    ],
    'source_plus_downstream':[
        'log_update_distance','log_state_distance',
        'normalized_nonlinear_residual','tangent_alignment',
        'endpoint_jacobian_disagreement','log_downstream_Q',
        'downstream_fisher_gain_full','downstream_residual_fraction_Q'
    ]
}

if confirm_pairs is None:
    print('LORO pending.')

else:
    q=prep(confirm_pairs[confirm_pairs.condition.eq(PRIMARY_CONDITION)]).copy()
    q['target']=np.log1p(q.KL.clip(lower=0))

    repos=sorted(q.repo.unique())

    # Leave-one-run-out requires at least TWO independent pretrained runs:
    # one held out for testing and at least one remaining for training.
    # Smoke mode intentionally uses only one run, so LORO is undefined there.
    if len(repos) < 2:
        print(
            f'LORO skipped: only {len(repos)} pretrained run is available '
            f'({repos[0] if repos else "none"}). '
            'This is expected in smoke mode. '
            'Run full_confirmatory (6 runs) for the actual LORO analysis.'
        )
        loro=pd.DataFrame(columns=[
            'heldout_repo','feature_set','n_train','n_test','spearman'
        ])
        loro.to_csv(
            OUT_DIR/'LORO_SAFE_TRANSFER_PREDICTION.csv',
            index=False
        )

    else:
        scores=[]

        for held in repos:
            tr0=q[q.repo.ne(held)].copy()
            te0=q[q.repo.eq(held)].copy()

            for name,fs in FEATURES.items():
                tr=(
                    tr0[fs+['target']]
                    .replace([np.inf,-np.inf],np.nan)
                    .dropna()
                )
                te=(
                    te0[fs+['target']]
                    .replace([np.inf,-np.inf],np.nan)
                    .dropna()
                )

                # Defensive guard: never fit sklearn on an empty split.
                if len(tr)==0 or len(te)==0:
                    print(
                        f'Skipping {held} / {name}: '
                        f'n_train={len(tr)}, n_test={len(te)}'
                    )
                    scores.append({
                        'heldout_repo':held,
                        'feature_set':name,
                        'n_train':len(tr),
                        'n_test':len(te),
                        'spearman':np.nan,
                    })
                    continue

                m=make_pipeline(StandardScaler(),Ridge(alpha=1.0))
                m.fit(tr[fs],tr.target)
                pred=m.predict(te[fs])

                rho=(
                    float(spearmanr(pred,te.target).statistic)
                    if len(te) >= 2
                    else np.nan
                )

                scores.append({
                    'heldout_repo':held,
                    'feature_set':name,
                    'n_train':len(tr),
                    'n_test':len(te),
                    'spearman':rho,
                })

        loro=pd.DataFrame(scores)
        loro.to_csv(
            OUT_DIR/'LORO_SAFE_TRANSFER_PREDICTION.csv',
            index=False
        )

        valid=loro.dropna(subset=['spearman']).copy()

        if len(valid):
            piv=valid.pivot(
                index='heldout_repo',
                columns='feature_set',
                values='spearman'
            )
            display(piv)

            if {
                'source_geometry',
                'source_plus_downstream'
            }.issubset(piv.columns):
                delta=(
                    piv['source_plus_downstream']
                    -piv['source_geometry']
                )
                print(
                    'Increment median',
                    float(delta.median()),
                    'positive',
                    int((delta>0).sum()),
                    '/',
                    len(delta),
                    'p',
                    exact_sign_p(delta.to_numpy())
                )
        else:
            print('No valid LORO folds after finite-value filtering.')


## 14. Safe-transfer taxonomy and lexical controls

In [ ]:
if confirm_pairs is None: print('Safe taxonomy pending.')
else:
    gg=confirm_pairs[confirm_pairs.condition.eq('global')]; q5=confirm_pairs[confirm_pairs.condition.eq(PRIMARY_CONDITION)]; parts=[]
    for repo,g in q5.groupby('repo'):
        gm=float(gg[gg.repo.eq(repo)].KL.median())
        for a in SAFE_ALPHA_SWEEP:
            h=g.copy(); h['alpha']=a; h['global_median_KL']=gm; h['safe_threshold_KL']=a*gm; h['is_safe']=(h.KL<=a*gm)&(h.top1_flip==0); parts.append(h)
    safe_df=pd.concat(parts,ignore_index=True); safe_df.to_csv(OUT_DIR/'SAFE_TRANSFER_LABELS_THRESHOLD_SWEEP.csv',index=False); ps=safe_df[np.isclose(safe_df.alpha,SAFE_ALPHA_PRIMARY)]
    tax=[]
    for repo,g in ps.groupby('repo'):
        nm=g.normalized_nonlinear_residual.median(); dm=g.downstream_fisher_gain_full.median(); h=g.copy(); h['source_high']=h.normalized_nonlinear_residual>=nm; h['downstream_high']=h.downstream_fisher_gain_full>=dm
        for (sh,dh),z in h.groupby(['source_high','downstream_high']): tax.append({'repo':repo,'source_mismatch':'high' if sh else 'low','downstream_visibility':'high' if dh else 'low','n':len(z),'safe_rate':float(z.is_safe.mean()),'median_KL':float(z.KL.median())})
    tax=pd.DataFrame(tax); tax.to_csv(OUT_DIR/'SOURCE_DOWNSTREAM_TAXONOMY.csv',index=False); display(tax.groupby(['source_mismatch','downstream_visibility'])[['safe_rate','median_KL']].median())
    lex=[]
    for cond in ['same_current_token','same_next_token','knn_5','global']:
        for repo,g in confirm_pairs[confirm_pairs.condition.eq(cond)].groupby('repo'):
            gm=float(gg[gg.repo.eq(repo)].KL.median()); safe=(g.KL<=SAFE_ALPHA_PRIMARY*gm)&(g.top1_flip==0); lex.append({'repo':repo,'condition':cond,'n':len(g),'median_KL':float(g.KL.median()),'no_flip_rate':float((g.top1_flip==0).mean()),'safe_rate':float(safe.mean())})
    lex=pd.DataFrame(lex); lex.to_csv(OUT_DIR/'LEXICAL_PORTABILITY_CONTROLS.csv',index=False); display(lex.groupby('condition')[['median_KL','no_flip_rate','safe_rate']].median())

## 15. Held-out-B source-tangent repair vs downstream-Fisher causal rotation

In [ ]:
if confirm_pairs is None: print('Repair comparison pending.')
else:
    q=confirm_pairs[confirm_pairs.condition.eq(PRIMARY_CONDITION)]; rows=[]
    for repo,g in q.groupby('repo'):
        row={'repo':repo,'n':len(g),'donor_KL':float(g.KL.median()),'source_tangent_KL':float(g.KL_source_tangent_repair.median()),'source_tangent_gain':float(g.source_tangent_repair_gain.median())}
        if 'KL_fisher_rotation' in g.columns: row.update({'fisher_rotation_KL':float(g.KL_fisher_rotation.median()),'fisher_rotation_gain':float(g.fisher_rotation_gain.median()),'random_KL':float(g.KL_angle_matched_random_mean.median()),'fisher_vs_random_gain':float(g.fisher_vs_random_gain.median())})
        rows.append(row)
    rr=pd.DataFrame(rows); rr.to_csv(OUT_DIR/'HELDOUT_B_REPAIR_RUN_EFFECTS.csv',index=False); display(rr)
    if 'fisher_rotation_gain' in rr.columns:
        v=rr.fisher_rotation_gain.to_numpy(); vr=rr.fisher_vs_random_gain.to_numpy(); print('Fisher repair median',float(np.median(v)),'positive',int((v>0).sum()),'/',len(v),'p',exact_sign_p(v)); print('vs random median',float(np.median(vr)),'positive',int((vr>0).sum()),'/',len(vr),'p',exact_sign_p(vr))

## 16. Predict which source-tangent repairs will help

In [ ]:
if confirm_pairs is None: print('Repair-gain prediction pending.')
else:
    q=confirm_pairs[confirm_pairs.condition.eq(PRIMARY_CONDITION)]; rows=[]
    for repo,g in q.groupby('repo'):
        rows.append({'repo':repo,'rho_predicted_vs_actual_tangent_gain':float(spearmanr(g.predicted_source_tangent_repair_gain,g.source_tangent_repair_gain).statistic)})
    gp=pd.DataFrame(rows); gp.to_csv(OUT_DIR/'REPAIR_GAIN_PREDICTION.csv',index=False); display(gp); v=gp.rho_predicted_vs_actual_tangent_gain.to_numpy(); print('median',float(np.nanmedian(v)),'positive',int((v>0).sum()),'/',len(v),'p',exact_sign_p(v))

## 17. Finite-dose falsification of the local Fisher approximation

In [ ]:
if confirm_dose is None and (OUT_DIR/'CONFIRMATORY_B_DOSE.csv').exists(): confirm_dose=pd.read_csv(OUT_DIR/'CONFIRMATORY_B_DOSE.csv')
if confirm_dose is None or len(confirm_dose)==0: print('Dose test pending.')
else:
    rows=[]
    for (repo,lam),g in confirm_dose.groupby(['repo','lambda']):
        rows.append({'repo':repo,'lambda':lam,'rho_predicted_actual':float(spearmanr(g.predicted_Fisher_KL,g.KL).statistic),'median_actual_to_predicted':float(np.median(g.KL/np.maximum(g.predicted_Fisher_KL,1e-12))),'actual_KL_median':float(g.KL.median()),'predicted_KL_median':float(g.predicted_Fisher_KL.median())})
    de=pd.DataFrame(rows); de.to_csv(OUT_DIR/'FISHER_LINEARITY_DOSE_AUDIT.csv',index=False); display(de.groupby('lambda')[['rho_predicted_actual','median_actual_to_predicted']].median())

## 18. Bidirectionality: test protocol dependence directly

In [ ]:
if confirm_pairs is None: print('Bidirectionality pending.')
else:
    q=confirm_pairs[confirm_pairs.condition.eq(PRIMARY_CONDITION)]; gg=confirm_pairs[confirm_pairs.condition.eq('global')]; rows=[]
    for repo,g in q.groupby('repo'):
        gm=float(gg[gg.repo.eq(repo)].KL.median())
        for a in SAFE_ALPHA_SWEEP:
            f=(g.KL<=a*gm)&(g.top1_flip==0); r=(g.reverse_KL<=a*gm)&(g.reverse_top1_flip==0); rows.append({'repo':repo,'alpha':a,'n':len(g),'forward_safe_rate':float(f.mean()),'reverse_safe_rate':float(r.mean()),'bidirectional_safe_rate':float((f&r).mean()),'reverse_given_forward':float(r[f].mean()) if f.any() else np.nan,'direction_disagreement_rate':float((f!=r).mean())})
    sym=pd.DataFrame(rows); sym.to_csv(OUT_DIR/'SAFE_TRANSFER_BIDIRECTIONALITY.csv',index=False); display(sym.groupby('alpha').median(numeric_only=True))

## 19. Transitivity: should we call them equivalence classes?

In [ ]:
if confirm_relations is None and (OUT_DIR/'CONFIRMATORY_B_RELATION_EDGES.csv').exists(): confirm_relations=pd.read_csv(OUT_DIR/'CONFIRMATORY_B_RELATION_EDGES.csv')
if confirm_relations is None or len(confirm_relations)==0 or confirm_pairs is None: print('Transitivity pending.')
else:
    gg=confirm_pairs[confirm_pairs.condition.eq('global')]; rows=[]
    for repo,g in confirm_relations.groupby('repo'):
        gm=float(gg[gg.repo.eq(repo)].KL.median())
        for a in SAFE_ALPHA_SWEEP:
            h=g.copy(); h['safe']=(h.KL<=a*gm)&(h.top1_flip==0); anteced=succ=viol=pairn=pairagree=0
            for _,t in h.groupby('triangle_id'):
                edge={(int(r.source_index),int(r.donor_index)):bool(r.safe) for _,r in t.iterrows()}; nodes=sorted(set(t.source_index).union(t.donor_index))
                for x,y in itertools.combinations(nodes,2):
                    if (x,y) in edge and (y,x) in edge: pairn+=1; pairagree+=int(edge[(x,y)]==edge[(y,x)])
                for x,y,z in itertools.permutations(nodes,3):
                    if (x,y) in edge and (y,z) in edge and (x,z) in edge and edge[(x,y)] and edge[(y,z)]:
                        anteced+=1; succ+=int(edge[(x,z)]); viol+=int(not edge[(x,z)])
            rows.append({'repo':repo,'alpha':a,'triangles':h.triangle_id.nunique(),'antecedents':anteced,'transitivity_rate':succ/anteced if anteced else np.nan,'violations':viol,'pair_relation_agreement':pairagree/pairn if pairn else np.nan})
    trans=pd.DataFrame(rows); trans.to_csv(OUT_DIR/'SAFE_TRANSFER_TRANSITIVITY.csv',index=False); display(trans)

## 20. Developmental scope: does downstream compatibility drift during pretraining?

In [ ]:
developmental_pairs=None
if RUN_GPU and RUN_DEVELOPMENTAL:
    rng=np.random.default_rng(stable_seed('dev-hosts')); dev_hosts=np.sort(rng.choice(POOL_N,size=min(N_DEV,POOL_N),replace=False)); parts=[]
    for repo in PRIMARY_REPOS:
        for step in DEV_STEPS:
            if step==FINAL_STEP and confirm_pairs is not None:
                parts.append(confirm_pairs[(confirm_pairs.repo.eq(repo))&(confirm_pairs.condition.eq(PRIMARY_CONDITION))].copy()); continue
            print('\n=== DEV',repo,step,'==='); p,_,_,_=run_safe_transfer_target(repo,step,PRIMARY_LAYER,pool_B,pool_B_next,dev_hosts,'B_dev',False,False); parts.append(p[p.condition.eq(PRIMARY_CONDITION)])
    developmental_pairs=pd.concat(parts,ignore_index=True); developmental_pairs.to_csv(OUT_DIR/'developmental'/'DEVELOPMENTAL_SAFE_TRANSFER_PAIRS.csv',index=False)
elif (OUT_DIR/'developmental'/'DEVELOPMENTAL_SAFE_TRANSFER_PAIRS.csv').exists(): developmental_pairs=pd.read_csv(OUT_DIR/'developmental'/'DEVELOPMENTAL_SAFE_TRANSFER_PAIRS.csv')
if developmental_pairs is None: print('Developmental scope not run.')
else:
    ds=developmental_pairs.groupby(['repo','actual_step'],as_index=False).agg(median_source_mismatch=('normalized_nonlinear_residual','median'),median_downstream_Q=('downstream_fisher_Q_full','median'),median_downstream_gain=('downstream_fisher_gain_full','median'),median_KL=('KL','median')); ds.to_csv(OUT_DIR/'developmental'/'DEVELOPMENTAL_SUMMARY.csv',index=False); display(ds)

## 21. Existing corpus-A repair bridge — explicitly exploratory

In [ ]:
old5=old_pairs[(old_pairs.actual_step.eq(FINAL_STEP))&(old_pairs.condition.eq(PRIMARY_CONDITION))]
oldr=old_repairs[(old_repairs.actual_step.eq(FINAL_STEP))&(old_repairs.condition.eq(PRIMARY_CONDITION))]
od=oldr[oldr.family.eq('donor')][['repo','source_index','KL']].rename(columns={'KL':'KL_old_donor'})
ot=oldr[(oldr.family.eq('norm_residual_excision'))&np.isclose(oldr['lambda'],1.)][['repo','source_index','KL']].rename(columns={'KL':'KL_old_tangent'})
b=old5.merge(od,on=['repo','source_index']).merge(ot,on=['repo','source_index']); b['old_repair_gain']=b.KL_old_donor-b.KL_old_tangent
rows=[]
for repo,g in b.groupby('repo'): rows.append({'repo':repo,'rho_source_residual_vs_old_repair_gain':float(spearmanr(g.normalized_nonlinear_residual,g.old_repair_gain).statistic),'rho_alignment_vs_old_repair_gain':float(spearmanr(g.tangent_alignment,g.old_repair_gain).statistic)})
bridge=pd.DataFrame(rows); bridge.to_csv(OUT_DIR/'EXPLORATORY_OLD_REPAIR_GAIN_SOURCE_METRICS.csv',index=False); display(bridge)

## 22. Optional Pythia scale sanity

In [ ]:
if RUN_GPU and RUN_SCALE_CHECKS:
    relative_depth=PRIMARY_LAYER/(6-1); rows=[]
    for repo in SCALE_REPOS:
        temp=load_model(repo,FINAL_STEP); layer=int(round((len(model_layers(temp))-1)*relative_depth)); del temp; clear_memory()
        p,_,_,_=run_safe_transfer_target(repo,FINAL_STEP,layer,pool_B,pool_B_next,HOST_IDX_CONFIRM[:min(128,len(HOST_IDX_CONFIRM))],'B_scale',False,False); q=p[p.condition.eq(PRIMARY_CONDITION)]; rows.append({'repo':repo,'layer':layer,'n':len(q),'median_KL':float(q.KL.median()),'median_source_mismatch':float(q.normalized_nonlinear_residual.median()),'median_downstream_Q':float(q.downstream_fisher_Q_full.median()),'median_downstream_gain':float(q.downstream_fisher_gain_full.median())})
    scale=pd.DataFrame(rows); scale.to_csv(OUT_DIR/'scope'/'PYTHIA_SCALE_SANITY.csv',index=False); display(scale)
else: print('Scale sanity disabled.')

## 23. Paper-ready figures

In [ ]:
if confirm_pairs is None: print('Held-out-B figures pending.')
else:
    runp=pd.read_csv(OUT_DIR/'PRIMARY_DOWNSTREAM_VISIBILITY_RUNS.csv'); fig,ax=plt.subplots(figsize=(5.8,3.5)); x=np.arange(len(runp)); ax.scatter(x,runp.partial_rho_Q_to_KL,s=45); ax.axhline(0,linewidth=1); ax.set_xticks(x,[r.split('/')[-1].replace('pythia-70m','base') for r in runp.repo],rotation=25,ha='right'); ax.set_ylabel('Partial Spearman with transplant KL'); ax.set_title('Downstream visibility beyond magnitude + source geometry'); fig.tight_layout(); fig.savefig(OUT_DIR/'figures'/'downstream_visibility_six_runs.pdf',bbox_inches='tight'); fig.savefig(OUT_DIR/'figures'/'downstream_visibility_six_runs.png',dpi=220,bbox_inches='tight'); plt.show()
    q=confirm_pairs[confirm_pairs.condition.eq(PRIMARY_CONDITION)]; fig,ax=plt.subplots(figsize=(5.6,4)); ax.scatter(q.normalized_nonlinear_residual,q.downstream_fisher_gain_full,s=10,alpha=.35); ax.set_xlabel('Source normalized nonlinear residual'); ax.set_ylabel('Downstream Fisher gain'); ax.set_title('Source incompatibility vs downstream visibility'); fig.tight_layout(); fig.savefig(OUT_DIR/'figures'/'source_vs_downstream_map.pdf',bbox_inches='tight'); fig.savefig(OUT_DIR/'figures'/'source_vs_downstream_map.png',dpi=220,bbox_inches='tight'); plt.show()
    rr=pd.read_csv(OUT_DIR/'HELDOUT_B_REPAIR_RUN_EFFECTS.csv'); fig,ax=plt.subplots(figsize=(6.2,3.7)); x=np.arange(len(rr)); ax.scatter(x,rr.donor_KL,label='Original donor',s=42); ax.scatter(x,rr.source_tangent_KL,label='Source-tangent repair',s=42,marker='D');
    if 'fisher_rotation_KL' in rr.columns: ax.scatter(x,rr.fisher_rotation_KL,label='Downstream-Fisher rotation',s=42,marker='s'); ax.scatter(x,rr.random_KL,label='Angle-matched random',s=42,marker='x')
    ax.set_xticks(x,[r.split('/')[-1].replace('pythia-70m','base') for r in rr.repo],rotation=25,ha='right'); ax.set_ylabel('Median transplant KL'); ax.set_title('Held-out-B causal repair comparison'); ax.legend(frameon=False); fig.tight_layout(); fig.savefig(OUT_DIR/'figures'/'heldout_B_repairs.pdf',bbox_inches='tight'); fig.savefig(OUT_DIR/'figures'/'heldout_B_repairs.png',dpi=220,bbox_inches='tight'); plt.show()

In [ ]:
# =========================
# FINAL CLEAN-RUN VERIFICATION
# =========================
_target_summary_path = OUT_DIR / "CONFIRMATORY_B_TARGET_SUMMARIES.csv"

if _target_summary_path.exists():
    _ts = pd.read_csv(_target_summary_path)
    display(_ts[["repo","pool_n","n_hosts","self_patch_mean_KL","self_patch_max_KL"]])

    assert len(_ts) == 6, f"Expected six confirmatory runs, got {len(_ts)}"
    assert (_ts["pool_n"] == 512).all(), _ts[["repo","pool_n"]]
    assert (_ts["n_hosts"] == 256).all(), _ts[["repo","n_hosts"]]

    print("CLEAN SIX-RUN CONFIRMATORY AUDIT: PASS")
    print("All 6 runs used pool=512 and n_hosts=256.")
else:
    print("Confirmatory target summary not written yet.")

## 24. Automatic claim gate

In [ ]:
claims=[{'claim':'Old source-tangent repair is a universal causal fix','status':'REJECTED_BY_EXISTING_3_OF_6_RESULT'}]
if confirm_pairs is None:
    claims += [{'claim':'Downstream visibility adds information beyond magnitude + source geometry','status':'PENDING_HELDOUT_B'},
               {'claim':'Downstream-Fisher rotation causally improves transfer','status':'PENDING_HELDOUT_B'},
               {'claim':'Safe transfer forms equivalence-like neighborhoods','status':'PENDING_RELATION_TEST'}]
else:
    primary_ok=bool(primary_new.iloc[0].get('primary_success_6of6',False)); claims.append({'claim':'Downstream visibility adds information beyond magnitude + source geometry','status':'SUPPORTED_PRIMARY' if primary_ok else 'NOT_SUPPORTED_BY_FROZEN_PRIMARY'})
    rr=pd.read_csv(OUT_DIR/'HELDOUT_B_REPAIR_RUN_EFFECTS.csv')
    if 'fisher_rotation_gain' in rr.columns:
        repair_ok=len(rr)==6 and int((rr.fisher_rotation_gain>0).sum())==6; spec_ok=primary_ok and repair_ok and int((rr.fisher_vs_random_gain>0).sum())==6
        claims.append({'claim':'Same-norm downstream-Fisher rotation reduces KL','status':'SUPPORTED_GATEKEPT' if primary_ok and repair_ok else 'NOT_SUPPORTED_AS_GATEKEPT_CAUSAL'}); claims.append({'claim':'Fisher repair beats same-angle random rotation','status':'SUPPORTED_GATEKEPT' if spec_ok else 'NOT_PROMOTED'})
    tp=OUT_DIR/'SAFE_TRANSFER_TRANSITIVITY.csv'; sp=OUT_DIR/'SAFE_TRANSFER_BIDIRECTIONALITY.csv'
    if tp.exists() and sp.exists():
        t=pd.read_csv(tp); s=pd.read_csv(sp); t=t[np.isclose(t.alpha,SAFE_ALPHA_PRIMARY)]; s=s[np.isclose(s.alpha,SAFE_ALPHA_PRIMARY)]; mt=float(t.transitivity_rate.median()) if len(t) else np.nan; mr=float(s.reverse_given_forward.median()) if len(s) else np.nan
        status='APPROX_EQUIVALENCE_LIKE_DESCRIPTIVELY' if np.isfinite(mt) and np.isfinite(mr) and mt>=.8 and mr>=.8 else 'DIRECTED_PROTOCOL_DEPENDENT_COMPATIBILITY_NOT_EQUIVALENCE'
        claims.append({'claim':'Operational safe transfer behaves like an equivalence relation','status':status})
claim_gate=pd.DataFrame(claims); claim_gate.to_csv(OUT_DIR/'CLAIM_GATE.csv',index=False); display(claim_gate)

## 25. Manifest and recursive results ZIP

In [ ]:
manifest={'project':'safe MLP transfer falsification','mode':MODE,'protocol_sha256':protocol_hash,'existing_causal_zip':str(EXISTING_CAUSAL_ZIP),'existing_causal_zip_sha256':sha256_file(EXISTING_CAUSAL_ZIP),'main_results_zip':str(MAIN_RESULTS_ZIP),'main_results_zip_sha256':sha256_file(MAIN_RESULTS_ZIP),'primary_models':PRIMARY_REPOS,'heldout_corpus':'B','final_step':FINAL_STEP,'layer':PRIMARY_LAYER,'n_confirm_hosts':N_CONFIRM,'primary_condition':PRIMARY_CONDITION,'primary_endpoint':'partial Spearman downstream Fisher Q -> KL controlling magnitude + source geometry','replication_unit':'independently pretrained run','gpu_experiment_executed':RUN_GPU}
(OUT_DIR/'MANIFEST.json').write_text(json.dumps(manifest,indent=2))
inv=[]
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file(): inv.append({'relative_path':str(p.relative_to(OUT_DIR)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(inv).to_csv(OUT_DIR/'OUTPUT_INVENTORY.csv',index=False)
zout=OUT_DIR.parent/f'{OUT_DIR.name}__ALL_OUTPUTS.zip'
with zipfile.ZipFile(zout,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.rglob('*')):
        if p.is_file(): zf.write(p,arcname=p.relative_to(OUT_DIR.parent))
print('Results ZIP:',zout)

In [ ]:
# Colab convenience: the ZIP is already persisted in Drive when USE_GOOGLE_DRIVE=True.
if "google.colab" in sys.modules:
    print("Final results ZIP:", zout)
    print("If Drive persistence is enabled, you do not need to manually download it before disconnecting.")

## 26. Decision table after execution

- **Downstream Q survives controls in 6/6:** evidence that safe transfer depends on what the suffix reads, not only on source similarity.
- **Source geometry wins; downstream adds nothing:** falsifies the downstream-nullspace story.
- **Neither survives magnitude:** the constructive mechanism story is weak; perturbation size dominates.
- **Fisher rotation succeeds in 6/6 and beats random:** downstream geometry becomes an intervention rule, not just a diagnostic.
- **Safe relation is asymmetric/non-transitive:** call it a directed compatibility graph, **not** an equivalence class.
- **High bidirectionality + high transitivity:** evidence for genuine computational neighborhoods under the specified behavioral metric.

Every major story in this notebook can fail cleanly.